# Tour mode choice, work tours (article table est_mc_work)

A model of MAST (Maryland Activity Scheduling and Travel model), the activity-based model of the MILES platform, built on the ABIT code base.

## What this model answers

Which mode does a person use for work tours? The model compares the available modes by time, cost or distance and by person and household traits; car driver is the reference.

| | |
|---|---|
| Choice | The tour mode |
| Alternatives | car driver, walk, bike, car passenger, bus, train (as available) |
| Decision makers | Persons on work tours |
| Data | Regional Travel Survey 2017/18 (Baltimore and Washington regions), estimation half |

## How to run

- Kernel: *Python (biogeme)* (`~/.venvs/biogeme`, Biogeme 3.3). Then *Run All*.
- `RUN_DATA_PREP`: rebuild the records and tables in a scratch copy and compare them with the shipped ones.
- `RUN_CHECK`: log-likelihood at the reference coefficients, no estimation.
- `RUN_ESTIMATION`: run the Biogeme estimation (off by default).
- `RUN_HOLDOUT`: log-likelihood of the hold-out half.

## Contents

This notebook goes from the survey records to the estimates of this model:

1. Raw data: survey files, estimation and hold-out halves, the records of this model
2. Data preparation: filters from the survey to the records (records kept and dropped), rebuild check against the shipped tables
3. Choice set and estimation data
4. Descriptive statistics and figures; 4.1 every explanatory variable of the model in the data
5. Model specification (the folder's `model/estimate_tour_mode_work.py`, unchanged)
6. Correlation of the explanatory variables (heatmaps, VIF)
7. Log-likelihood at the reference coefficients
8. Biogeme estimation
9. Results: article format, what the coefficients say, fit, comparison with the article, correlation of the estimates
10. Hold-out sample
11. Glossary

## Model at a glance

**1. Who decides.** Persons on work tours. Data: Regional Travel Survey 2017/18 (Baltimore and Washington regions), estimation half; 9,126 tours.

**2. What is chosen (dependent variable).** The tour mode. Alternatives: car driver, walk, bike, car passenger, bus, train (as available).

**3. Explanatory variables** (signs of the article estimates; terms specific to an alternative or purpose are listed per series):

| Group | Variable | Sign |
|---|---|---|
| Level of service | Generalised cost (min) | All: − |
| Level of service | Car driver: paid-parking share at the destination | All: − |
| Level of service | Car driver: tour with stops | All: + |
| Constants and scales | Constant | Walk: +, Bike: −, Car passenger: −, Bus: −, Train: − |
| Person, household and destination | Round-trip distance (km) | Walk: −, Bike: −, Car passenger: −, Bus: −, Train: + |
| Person, household and destination | Age (years) | Walk: −, Bike: −, Car passenger: −, Bus: −, Train: − |
| Person, household and destination | Man | Walk: +, Bike: +, Car passenger: −, Bus: −, Train: − |
| Person, household and destination | Driving licence | Bike: +, Car passenger: −, Bus: −, Train: − |
| Person, household and destination | Household size | Walk: +, Bike: +, Car passenger: +, Bus: +, Train: − |
| Person, household and destination | Cars | Walk: −, Bike: −, Car passenger: +, Bus: −, Train: − |
| Person, household and destination | Member under 18 | Walk: −, Bike: −, Car passenger: −, Bus: −, Train: − |
| Person, household and destination | Core city | Walk: −, Bike: +, Car passenger: −, Bus: +, Train: + |
| Person, household and destination | Rural | Bike: −, Car passenger: +, Bus: −, Train: − |

**4. How person and household traits enter.** Traits enter with one coefficient per alternative, against the reference alternative whose utility is 0.

**5. How to read a coefficient.** Positive makes the alternative more likely, negative less likely; exp(coefficient) is the factor by which the odds change for one unit more.

In [ ]:
# switches (see How to run) and set-up: imports, folders, helper modules
RUN_DATA_PREP  = True    # rebuild the records and estimation tables in a scratch copy and compare with the shipped tables (about 3 min)
RUN_CHECK      = True    # log-likelihood at the reference coefficients
RUN_ESTIMATION = False   # Biogeme estimation, written to output/ (False: results are read from output/ if there are any)
RUN_HOLDOUT    = True    # log-likelihood of the hold-out half at the reference coefficients

import os, sys, json, glob, inspect, importlib, time, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt; import nb_reader as rd; import figs as fg; import abit_figs as af
from IPython.display import Markdown
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "eval"; MODEL_FOLDER = "14_tour_mode_work"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", os.path.basename(FOLDER))

## 1. Raw data

*Where the records come from. Look for: the files and the record counts.*

### 1.1 The 2017/18 Regional Travel Survey

All MAST models are estimated on the 2017/18 RTS of the Baltimore and Washington regions (weekday travel day). Three files
are shipped in `../00_shared_data/rts_2017_18/raw`:

- `RTS_raw_flat_data.csv.gz`: the survey as received, one row per trip with its household, person and vehicle fields;
- `rts_trips_clean.csv.gz`: the trips after the survey cleaning (one row per trip, departure / arrival in minutes);
- `rts_tours.csv.gz`: home-based tours, work-based subtours and open tours built from the clean trips.

The cleaning script of the last two files is not shipped; the cleaned files are the starting point of the preparation here.
Race and ethnicity fields are not read.

In [ ]:
# 1.1 The 2017/18 Regional Travel Survey
flat, trips, tours = nt.rts_raw()
split = pd.read_csv(os.path.join(nt.ABIT_SHARED, "rts_2017_18", "split", "rts_split.csv"))
display(nt.rts_counts(flat, trips, tours, split))
show = ["tripid", "household_id", "person_id", "tripno", "o_activity", "d_activity", "dep_min", "arr_min", "travel_mode",
        "distance", "hhsize", "numvehicle", "hh_income_broad", "age", "employment_status", "student_status", "wttrdfin"]
trips[[c for c in show if c in trips]].head()

### 1.2 Estimation and hold-out halves

The households are split in two halves (`make_rts_split.py`): strata of home county x household size (1 to 5+), a seeded
shuffle inside each stratum (seed 2026), then alternate assignment. Every model is estimated on the estimation half; the
hold-out (validation) half is the other half of the same survey and is not read by the estimation. With `RUN_DATA_PREP` the
split is rebuilt into a scratch folder and compared with the shipped `rts_split.csv`.

In [ ]:
# 1.2 Estimation and hold-out halves
REBUILD = os.path.join(OUT, "rebuild_check")            # scratch folder: provided files are never overwritten
if RUN_DATA_PREP:
    rc = nt.run_quiet(f"{sys.executable} make_rts_split.py", cwd=nt.ABIT_PREP, env={"ABIT_REBUILT": REBUILD})
    display(nt.compare_frames(pd.read_csv(os.path.join(REBUILD, "rts_2017_18", "split", "rts_split.csv")), split,
                              key=["household_id"], name="rts_split.csv"))
else:
    print("skipped (RUN_DATA_PREP = False)")
hh = flat.drop_duplicates("household_id").set_index("household_id")
hh["half"] = split.set_index("household_id")["half"]
t = pd.crosstab(hh["hhsize"].clip(upper=5).rename("household size (5 = 5+)"), hh["half"])
t.loc["all"] = t.sum()
t

### 1.3 Records of this model

`input/data/raw/tours_<half>.parquet` holds the home-based tours of both halves (one row per tour: purpose, main activity,
home and main zone, chosen mode, level of service of every mode); `gc_bio_<segment>_s50` / `gc_hold_<segment>_s50` are the
tours of one segment whose chosen mode is available, `wesml_est` / `wesml_hold` the WESML weights of the work tours.
This model uses the work tours.

In [ ]:
# 1.3 Records of this model
CODE = os.path.join(FOLDER, "input", "code"); RAW = os.path.join(FOLDER, "input", "data", "raw")
HALVES = ["estimation", "validation"]; GC = {"estimation": "gc_bio_work_s50", "validation": "gc_hold_work_s50"}
TOURS = {h: pd.read_parquet(os.path.join(RAW, f"tours_{h}.parquet")) for h in HALVES}
GCT = {h: pd.read_parquet(os.path.join(RAW, GC[h] + ".parquet")) for h in HALVES}
display(pd.concat({h: TOURS[h].groupby("segment").size() for h in HALVES}, axis=1).rename_axis("tours by segment"))
display(pd.crosstab(TOURS["estimation"].segment, TOURS["estimation"].choice2, margins=True).rename_axis(index="segment (estimation half)", columns="chosen mode"))
cols = ["person_id", "purpose", "type", "segment", "hz", "mz", "start", "end", "choice", "choice2", "joint", "km", "car_min",
        "bus_ok", "train_ok", "age", "lic", "hhsize", "autos", "inc_year", "w"]
TOURS["estimation"][cols].head()

## 2. Data preparation

*From the raw records to the estimation sample. Look for: the records kept and dropped at each filter. With RUN_DATA_PREP the rebuilt tables must equal the shipped ones.*

### 2.1 From survey episodes to tours

`../00_shared_prep/code/prep_tours.py` cuts the typed episodes of each person-day into home-based chains (a chain runs
from a departure from home to the next arrival at home), keeps chains that start and end at home, drops tours with an
air, water or other trip, keeps persons of the survey population and tours whose home and main zone are in the zone
system. The main activity is the work episode, else school, else the longest episode; the tour mode is the highest mode
used on any trip (transit > car driver > car passenger > bike > walk). With `RUN_DATA_PREP` the shared chain (episodes,
tours, segment tables{', WESML weights' if True else ''}) is rebuilt into a scratch folder; its step counts and the comparison
with the shipped files follow.

In [ ]:
# Data preparation
REBUILD = os.path.join(OUT, "rebuild_check")
if RUN_DATA_PREP:
    t0 = time.time()
    ENV = {"ABIT_REBUILT": REBUILD, "WESML_TOURS": os.path.join(REBUILD, "mode_choice_work")}
    nt.run(f"{sys.executable} prep_tours.py", cwd=nt.ABIT_PREP, env=ENV, quiet=True)
    nt.run(f"{sys.executable} prep_tours.py biodata2", cwd=nt.ABIT_PREP, env=ENV, quiet=True)
    nt.run(f"{sys.executable} wesml_targets.py", cwd=nt.ABIT_PREP, env=ENV, quiet=True)
    st = {h: pd.read_csv(os.path.join(REBUILD, "mode_choice_inputs", f"tour_build_{h}.csv")) for h in HALVES}
    def kept(d):
        out, n = [], None
        for step, k in zip(d.step, d.n):
            n = n - k if step.startswith("dropped") else k
            out.append((step.replace("dropped: ", "without "), n))
        return out
    display(nt.funnel2({h: kept(st[h]) for h in HALVES},
                       {"without air / water / other mode": "tour with an air, water or other trip",
                        "persons of the survey-population input": "person not in the survey population",
                        "home and main zone in the zone system": "home or main-activity tract outside the zone system"}))
    W_ = os.path.join(REBUILD, "mode_choice_work")
    pairs = [(os.path.join(REBUILD, "rts_2017_18", "episodes", f"episodes_{h}.parquet"), os.path.join(nt.ABIT_SHARED, "rts_2017_18", "episodes", f"episodes_{h}.parquet")) for h in HALVES]
    pairs += [(os.path.join(W_, f"tours_{h}.parquet"), os.path.join(RAW, f"tours_{h}.parquet")) for h in HALVES]
    pairs += [(os.path.join(W_, GC[h] + ".parquet"), os.path.join(RAW, GC[h] + ".parquet")) for h in HALVES]
    pairs += [(os.path.join(REBUILD, "mode_choice_inputs", "parking_paid_share.csv"), os.path.join(nt.ABIT_SHARED, "mode_choice_inputs", "parking_paid_share.csv"))]
    pairs += [(os.path.join(REBUILD, "mode_choice_inputs", "wesml", f), os.path.join(RAW, f)) for f in ["wesml_est.parquet", "wesml_hold.parquet"] if os.path.exists(os.path.join(RAW, f))]
    res_shared = pd.concat([nt.compare_frames(nt.read_table(a), nt.read_table(b), name=os.path.basename(b)) for a, b in pairs], ignore_index=True)
    print(f"shared chain rebuilt in {(time.time() - t0) / 60:.1f} min")
    display(res_shared)
else:
    print("skipped (RUN_DATA_PREP = False)")

### 2.2 From tours to the estimation table

The model keeps the tours of segment *work* whose chosen mode is available (availability rules in section 3); the shipped
segment table `gc_bio_work_s50` is that selection, and `build_estimation_tables.py` keeps its numeric columns
and adds the WESML weight (`input/data/estimation/tours.parquet`). Tours dropped because the
chosen mode was not available, by mode:

In [ ]:
# 2.2 From tours to the estimation table
steps = {}
for h in HALVES:
    t = TOURS[h]
    steps[h] = [("home-based tours", len(t)), ("tours of the segment", int((t.segment == 'work').sum())),
                ("chosen mode available (estimation table)", len(GCT[h]))]
display(nt.funnel2(steps, {"tours of the segment": "other segments", "chosen mode available (estimation table)": "chosen mode not available"}))
drop = pd.concat({h: TOURS[h][TOURS[h].segment == 'work'].choice2.value_counts() - GCT[h].choice.value_counts().reindex(TOURS[h][TOURS[h].segment == 'work'].choice2.value_counts().index).fillna(0) for h in HALVES}, axis=1)
drop.astype(int).rename_axis("chosen mode").rename(columns=lambda c: f"{c}: dropped")

### 2.3 Rebuild check of the estimation table

`build_estimation_tables.py` is run in a scratch copy of `input/code` whose `input/data/raw` holds the rebuilt tours and
segment tables of 2.1; the rebuilt `tours.parquet` of both halves is compared with the shipped one.

In [ ]:
# 2.3 Rebuild check of the estimation table
if RUN_DATA_PREP:
    import shutil
    SCR = nt.scratch_tree(FOLDER)
    W_ = os.path.join(REBUILD, "mode_choice_work")
    for f in os.listdir(RAW):
        src = os.path.join(W_, f) if os.path.exists(os.path.join(W_, f)) else os.path.join(REBUILD, "mode_choice_inputs", "wesml", f)
        if f.endswith(".parquet") and os.path.exists(src):
            shutil.copy2(src, os.path.join(SCR, "input", "data", "raw", f))
    print("raw files of the scratch copy (rebuilt):", sorted(os.listdir(os.path.join(SCR, "input", "data", "raw"))))
    nt.run(f"{sys.executable} build_estimation_tables.py", cwd=os.path.join(SCR, "input", "code"))
    res = pd.concat([nt.compare_trees(os.path.join(SCR, "input", "data", h), os.path.join(FOLDER, "input", "data", h)).assign(half=h) for h in ["estimation", "holdout"]], ignore_index=True)
    nt.drop_scratch(SCR)
    display(res)
    nt.rebuild_verdict(pd.concat([res_shared, res], ignore_index=True), glob.glob(os.path.join(FOLDER, "model", "estimate_*.py"))[0])
else:
    print("skipped (RUN_DATA_PREP = False)")

## 3. Choice set and estimation table

*The table the model reads. Look for: the columns used and what they mean (data dictionary).*

Six alternatives (code `y`): 1 car driver, 2 walk, 3 bike, 4 car passenger, 5 bus, 6 train. Availability: car driver if the
person is 16+, has a licence and the household has a car (and the household's cars are not all used by other members'
overlapping car-driver tours); car passenger always; walk if the round trip is at most 12 km, bike at most 50 km; bus and
train if a transit path exists (walk access; drive access at the home end for train) and home and destination differ.
Level of service: car time and cost from the period skims, transit time and fare, generalised cost `gc_*` (minutes, value
of time 50 % of the wage rate).

In [ ]:
# Choice set and estimation table
LAB = {1: "car driver", 2: "walk", 3: "bike", 4: "car passenger", 5: "bus", 6: "train"}
EST_T = pd.read_parquet(os.path.join(FOLDER, "input", "data", "estimation", "tours.parquet"))
HOLD_T = pd.read_parquet(os.path.join(FOLDER, "input", "data", "holdout", "tours.parquet"))
print(f"estimation table: {EST_T.shape[0]:,} rows x {EST_T.shape[1]} columns; hold-out {HOLD_T.shape[0]:,} rows")
av = [c for c in EST_T.columns if c.startswith("av_")]
t = pd.DataFrame({"available (% of records)": (100 * (EST_T[av] > 0).mean()).round(1)})
t.index = [c[3:] for c in av]
display(t.T)
display(EST_T.head())
dd = os.path.join(FOLDER, "input", "data", "estimation", "data_dictionary.csv")
pd.read_csv(dd) if os.path.exists(dd) else None

## 4. Descriptive statistics

*What the sample looks like. Look for: plausible shares and maps, and the chosen alternatives against the available ones.*

### Observed mode shares (unweighted, survey-weighted and WESML-weighted)

In [ ]:
# Observed mode shares of the estimation half; the survey weight (and the WESML weight for work tours) re-weight
assert len(GCT["estimation"]) == len(EST_T) and (GCT["estimation"].y.to_numpy() == EST_T.y.to_numpy()).all()   # row-aligned

sh = nt.shares_table(EST_T.y.map(LAB), None if False else GCT["estimation"].w)
if "wesml" in EST_T:
    ws = EST_T.groupby(EST_T.y.map(LAB)).wesml.sum(); sh["WESML-weighted share %"] = (100 * ws / ws.sum()).round(2)
sh = sh.reindex([v for v in LAB.values() if v in sh.index])
display(sh)
cols_ = [c for c in sh.columns if "share" in c]
nt.grouped_bars(sh[cols_].rename(columns={"share %": "unweighted", "weighted share %": "survey weight", "WESML-weighted share %": "WESML weight"}), "chosen mode", "share of tours (%)", colors=[nt.OBS, nt.LIGHT, "#8a8a8a"], rot=0)
plt.tight_layout(); plt.show()

Observed mode shares of the estimation half; the survey weight (and the WESML weight for work tours) re-weights them to the population.

In [ ]:
# Round-trip car distance by chosen mode (box: quartiles, line: median, whiskers: 1.5 interquartile range); walk
fig, ax = nt.panels(2, ncol=2, w=6, h=3)
labs = [lab for m, lab in LAB.items() if (EST_T.y == m).any()]
data = [EST_T.km[EST_T.y == m].clip(lower=0.5) for m, lab in LAB.items() if (EST_T.y == m).any()]
bp = ax[0].boxplot(data, vert=False, tick_labels=[f"{l} ({len(d)})" for l, d in zip(labs, data)], showfliers=False, patch_artist=True,
                   medianprops=dict(color="white"), boxprops=dict(facecolor=nt.OBS, color=nt.OBS), whiskerprops=dict(color=nt.OBS), capprops=dict(color=nt.OBS))
ax[0].set_xscale("log"); ax[0].set_xlabel("round-trip car distance (km, log scale)"); ax[0].invert_yaxis()
b = nt.binned(EST_T.km, [0, 5, 10, 20, 40, 80, 1e9], ["0-5", "5-10", "10-20", "20-40", "40-80", "80+"])
nt.bar(b, "round-trip car distance (km)", "share of records (%)", ax=ax[1])
plt.tight_layout(); plt.show()
print(f"median round-trip distance {EST_T.km.median():.1f} km; by mode:", EST_T.groupby(EST_T.y.map(LAB)).km.median().round(1).to_dict())

Round-trip car distance by chosen mode (box: quartiles, line: median, whiskers: 1.5 interquartile range); walk and bike tours are the shortest.

In [ ]:
# Explanatory variables of the records: household, person and tour (the representative member for joint groups)
inc = pd.cut(EST_T.inc_year, [-1, 50000, 100000, 150000, 1e12], labels=["<50k", "50-100k", "100-150k", "150k+"])
items = [(EST_T.hhsize.clip(upper=6), "household size (6 = 6+)"), (EST_T.autos.clip(upper=4), "cars (4 = 4+)"),
         (inc, "household income ($/year)"), (EST_T.lic, "driving licence (1 yes)"), (EST_T.kids, "member under 18 (1 yes)"),
         (EST_T.stops.clip(upper=4), "stops on the tour (4 = 4+)")]
fig, ax = nt.panels(len(items) + 2, ncol=4, w=3.4, h=2.7)
for a, (v, lab) in zip(ax, items):
    c = v.value_counts().sort_index(); nt.bar(100 * c / c.sum(), lab, "share (%)", ax=a, rot=30 if "income" in lab else 0)
nt.hist(EST_T.age, "age (years)", "records", bins=range(0, 96, 5), ax=ax[-2])
nt.hist(EST_T.t_CD.clip(upper=180), "car time of the tour (min)", "records", bins=40, ax=ax[-1])
plt.tight_layout(); plt.show()

Explanatory variables of the records: household, person and tour (the representative member for joint groups).

In [ ]:
# Period of the main activity, and home and main-activity zones of the tours (Baltimore-Washington area)
g = GCT["estimation"]
per = g.pout.map({0: "AM 6-9", 1: "midday 9-15", 2: "PM 15-19", 3: "night"}).value_counts().reindex(["AM 6-9", "midday 9-15", "PM 15-19", "night"])
ZA = pd.read_csv(os.path.join(nt.ABIT_SHARED, "zones", "zones_model_attributes.csv")).sort_values("idx").reset_index(drop=True)
nt.bar(100 * per / per.sum(), "period of the main activity", "share of tours (%)", rot=0, figsize=(5.5, 2.8)); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(1, 2, figsize=(13, 5.6))
nt.zone_map(pd.Series(g.groupby("h").size().pipe(lambda s: pd.Series(s.values, index=ZA.id.values[s.index]))), "tours (home zone)", ax=ax[0], log=True)
nt.zone_map(pd.Series(g.groupby("m").size().pipe(lambda s: pd.Series(s.values, index=ZA.id.values[s.index]))), "tours (main-activity zone)", ax=ax[1], log=True)
ax[0].set_title("home zones", fontsize=10); ax[1].set_title("main-activity zones", fontsize=10)
plt.tight_layout(); plt.show()
tr = g.groupby(g.m.map(lambda i: ZA.countyfips.iat[i]).pipe(nt.area_group_of_county), observed=False).y.apply(lambda y: 100 * (y >= 5).mean())
print("transit share (%) by area group of the main-activity zone:", tr.round(1).to_dict())

Period of the main activity, and home and main-activity zones of the tours (Baltimore-Washington area).

### WESML weights

Work tours are re-weighted to the ACS 2017-19 commute-mode shares of 23 county strata: w = Q(mode | stratum) / H(mode | stratum),
Q from the ACS, H the unweighted share of the survey tours. The weights enter the log-likelihood (w · ln P).

In [ ]:
# The survey over-represents train commuters against the ACS: their weights are below 1, those of bus and car pa
WE = pd.read_parquet(os.path.join(RAW, "wesml_est.parquet"))
print(f"WESML weight: mean {EST_T.wesml.mean():.3f}, min {EST_T.wesml.min():.3f}, max {EST_T.wesml.max():.3f}, {WE.stratum.nunique()} strata")
display(EST_T.groupby(EST_T.y.map(LAB)).wesml.agg(["mean", "min", "max"]).round(3).rename_axis("mode"))
tg = pd.read_csv(os.path.join(nt.ABIT_SHARED, "mode_choice_inputs", "wesml", "wesml_targets.csv"))
tg = tg[tg.half.eq("est")]
tot = tg.drop_duplicates("stratum").n_tours.sum(); q = tg.assign(x=tg.acs * tg.n_tours).groupby("mode").x.sum() / tot
hh_ = tg.assign(x=tg.survey_unweighted * tg.n_tours).groupby("mode").x.sum() / tot
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
nt.hist(EST_T.wesml, "WESML weight", "work tours", bins=50, ax=ax[0])
nt.grouped_bars(pd.DataFrame({"ACS commute share": 100 * q, "survey tours (unweighted)": 100 * hh_}), "mode", "share (%)", ax=ax[1], colors=[nt.LIGHT, nt.OBS])
plt.tight_layout(); plt.show()

The survey over-represents train commuters against the ACS: their weights are below 1, those of bus and car passenger above 1.

### Estimation half vs hold-out half

In [ ]:
# Unweighted mode shares of the two halves
t = nt.half_comparison(EST_T.y.map(LAB), HOLD_T.y.map(LAB), "chosen mode")
print(f"records: estimation {len(EST_T):,}, hold-out {len(HOLD_T):,}; largest difference of a share: {t['difference (points)'].abs().max():.1f} points")
t

Unweighted mode shares of the two halves.

### 4.1 Variables in the data

Each explanatory variable of the final specification as it enters the utility, interactions included (read from the model's own utility functions: the part of the utility that each coefficient multiplies; coefficients that share a label, e.g. the copies of a term for several purposes, are shown in one panel). Here: each variable by the chosen alternative, estimation half, unweighted. Person and household variables (the same value in every utility they enter) are shown as the mean or share among the records that chose each alternative; attributes of the alternatives (values that differ between alternatives, e.g. cost) as the mean at the chosen alternative against the mean where the alternative is available.

In [ ]:
# every explanatory variable of the specification by chosen alternative
import varplots as vp
MV_ = af.module(MODEL_FOLDER); r_ = MV_.read("estimation"); D_ = r_[0] if isinstance(r_, tuple) else r_
e_ = MV_.formulas() if hasattr(MV_, "formulas") else MV_.loglike(); e_ = e_["loglike"] if isinstance(e_, dict) else e_
U_, AV_, CH_ = vp.find_choice(e_); ALT_NAMES = af.ALTS[MODEL_FOLDER]
VP = vp.by_choice(U_, AV_, CH_, D_, vp.abit_labels(MODEL_FOLDER), ALT_NAMES)
display(Markdown(vp.choice_counts_line(vp.choices(CH_, D_), ALT_NAMES, 'tours')))
vp.by_choice_figures(VP, list(ALT_NAMES.values()), unit='tours')

How to read it: dark grey bars = mean (or share with value 1) among the records that chose the alternative, grey range = 25th-75th percentile, dashed line = all records; for attributes of the alternatives, the light bar is the mean where the alternative is available.

### Data sources

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**2017/18 Regional Travel Survey (Baltimore and Washington regions), household travel survey, weekday travel day** — Baltimore Metropolitan Council (BMC) and the National Capital Region Transportation Planning Board (TPB) at the Metropolitan Washington Council of Governments (MWCOG)
- Where: https://www.baltometro.org/ and https://www.mwcog.org/ (verify: no download page is documented in the project; the data are released on request)
- How: Request the 2017/18 Regional Travel Survey public-use files from BMC (Baltimore region) or MWCOG/TPB (Washington region). The project used one flat file in which every trip row carries its household and person fields (household, person, trip and vehicle tables joined): RTS_raw_flat_data.csv, with the codebook RTS_codebook_data_dictionary.xlsx (sheets Household, Person, Trip, Vehicle). The two cleaned files rts_trips_clean.csv (163,290 trips, deduplicated) and rts_tours.csv (57,627 home-based tours) were made by 01_clean_rts.py of the tour-based MITO project; that script is not on this machine, so these two files are shipped and cannot be rebuilt here. Survey weights: wthhfin, wtperfin, wttrdfin. If you receive separate household / person / trip / vehicle tables instead, join them on household_id, person_id (persno) and tripid to the flat layout of the codebook.
- Put in: 00_shared_data/raw/rts_2017_18 (the project's copies are provided in 00_shared_data/rts_2017_18/raw as .csv.gz)
- Files: `RTS_raw_flat_data.csv`, `RTS_codebook_data_dictionary.xlsx`, `rts_trips_clean.csv`, `rts_tours.csv`

**Maryland Statewide Transportation Model (MSTM) 2016 skims, 1,588 SMZ/RMZ zones** — Maryland Department of Transportation (MDOT SHA), via the project's MAST Maryland set-up (ABIT/input/mito_los/skims)
- Where: https://www.roads.maryland.gov/ (verify: MSTM data are provided by MDOT on request; no download page is documented)
- How: Obtain the MSTM 2016 skims in OMX format (matrix mat1, lookups origins / destinations): traveltime_auto.omx (AM-peak car time, min), distance_auto.omx (car distance, m), bus_tt.omx (bus time, min; 1000 = no path, 5 intrazonal), rail_tt00.omx (rail time, min; 0 = no rail path). convert_mstm_skims.py turns them into the csv.gz files the models read.
- Put in: 00_shared_data/raw/mstm_skims_2016
- Files: `traveltime_auto.omx`, `distance_auto.omx`, `bus_tt.omx`, `rail_tt00.omx`

**MSTM zone system: SMZ/RMZ polygons, zone table, MAST zone attributes** — ABIT/input/maryland/zone_attributes.csv (MAST Maryland set-up)
- Where: https://www.roads.maryland.gov/ (verify: provided with the MSTM)
- How: Zone polygons SMZ_RMZ_02152011inMSTM_EPSG26918.shp (+ .dbf .shx .prj; field id / SMZRMZ, 1,588 zones), zoneSystem.csv (ZoneId, COUNTYFIPS, ...), zone_attributes.csv (id, centroid x/y EPSG:26985, countyfips, region types, bbsr, area_km2;
- Put in: 00_shared_data/raw/mstm_zone_system
- Files: `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.dbf/.shx/.prj`, `zoneSystem.csv`, `zone_attributes.csv`

**Census 2010 Gazetteer file of census tracts (internal points)** — U.S. Census Bureau
- Where: https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html (verify: the project copy I695_pricing/data/census/Gaz_tracts_national.txt has no download log)
- How: 2010 Gazetteer, Census Tracts, national file Gaz_tracts_national.txt (GEOID, INTPTLAT, INTPTLONG). Used by survey_population.tract_zone: zone of each tract's internal point.
- Put in: 00_shared_data/raw/census_gazetteer_2010
- Files: `Gaz_tracts_national.txt`

**MTA Maryland GTFS static feeds: local bus, light rail, metro, MARC, commuter bus** — Maryland Transit Administration
- Where: https://feeds.mta.maryland.gov/gtfs/<feed> for feed = local-bus, light-rail, metro, marc, commuter-bus (listed on https://www.mta.maryland.gov/developer-resources)
- How: Run 00_shared_prep/code/fetch_gtfs.sh (curl, one zip per feed, download log). The project used the feeds of 2026-10-21 (local bus valid 2026-08-21 to 2027-02-06).
- Put in: 00_shared_data/raw/gtfs/mta_gtfs
- Files: `mta_local-bus.zip`, `mta_light-rail.zip`, `mta_metro.zip`, `mta_marc.zip`, `mta_commuter-bus.zip`

**GTFS of the 13 regional operators (with the 5 MTA feeds the 18 feeds merged by merge_gtfs_full.py)** — each operator; Mobility Database mirror
- Where: charm_city_circulator: https://transportation.baltimorecity.gov/files/cccgtfs824zip (mirror https://files.mobilitydatabase.org/tld-288_1/latest.zip ; https://files.mobilitydatabase.org/ntd-30201/latest.zip ; https://files.mobilitydatabase.org/mdb-493/latest.zip); rta_central_maryland: http://data.trilliumtransit.com/gtfs/centralmarylandrta-md-us/centralmarylandrta-md-us.zip (mirror https://files.mobilitydatabase.org/mdb-489/latest.zip); anne_arundel_transit: https://www.aacounty.org/gtfs (mirror https://files.mobilitydatabase.org/ntd-30129/latest.zip); annapolis_transit: https://www.arcgis.com/sharing/rest/content/items/c307923d57ab4871aff79491b4a1bfd3/data (mirror https://files.mobilitydatabase.org/mdb-2285/latest.zip); harford_transit_link: https://files.mobilitydatabase.org/mdb-501/latest.zip (mirror https://github.com/mobilityequity/maryland-local-gtfs/raw/master/Harford_GTFS.zip); carroll_transit: https://files.mobilitydatabase.org/mdb-500/latest.zip (mirror https://github.com/mobilityequity/maryland-local-gtfs/raw/master/Carroll_GTFS.zip); wmata_rail: https://api.wmata.com/gtfs/rail-gtfs-static.zip (API key required) (mirror https://files.mobilitydatabase.org/mdb-1847/latest.zip); wmata_bus: https://api.wmata.com/gtfs/bus-gtfs-static.zip (API key required) (mirror https://files.mobilitydatabase.org/mdb-1846/latest.zip); ride_on: https://www.montgomerycountymd.gov/DOT-Transit/Resources/Files/GTFS/RideOnGTFS.zip (mirror https://files.mobilitydatabase.org/mdb-488/latest.zip); thebus_pg: https://data.trilliumtransit.com/gtfs/princegeorgescounty-md-us/princegeorgescounty-md-us.zip (mirror https://files.mobilitydatabase.org/mdb-477/latest.zip); amtrak: https://content.amtrak.com/content/gtfs/GTFS.zip (mirror https://files.mobilitydatabase.org/mdb-11/latest.zip); harbor_connector: http://data.trilliumtransit.com/gtfs/harborconnector-md-us/harborconnector-md-us.zip (mirror https://files.mobilitydatabase.org/mdb-618/latest.zip); towson_loop: https://passio3.com/towson/passioTransit/gtfs/google_transit.zip (mirror https://files.mobilitydatabase.org/mdb-1784/latest.zip)
- How: Run 00_shared_prep/code/fetch_gtfs_regional.sh (official URL first, then the Mobility Database mirror; download log with sha256). The script also fetches the Bethesda Circulator (https://files.mobilitydatabase.org/mdb-231/latest.zip), which merge_gtfs_full.py does not use.
- Put in: 00_shared_data/raw/gtfs/<agency>
- Files: `charm_city_circulator/charm_city_circulator_gtfs.zip`, `rta_central_maryland/rta_central_maryland_gtfs.zip`, `anne_arundel_transit/anne_arundel_transit_gtfs.zip`, `annapolis_transit/annapolis_transit_gtfs.zip`, `harford_transit_link/harford_transit_link_gtfs.zip`, `carroll_transit/carroll_transit_gtfs.zip`, `wmata_rail/wmata_rail_gtfs.zip`, `wmata_bus/wmata_bus_gtfs.zip`, `ride_on/ride_on_gtfs.zip`, `thebus_pg/thebus_pg_gtfs.zip`, `amtrak/amtrak_gtfs.zip`, `harbor_connector/harbor_connector_gtfs.zip`, `towson_loop/towson_loop_gtfs.zip`

**OpenStreetMap Maryland extract** — Geofabrik
- Where: https://download.geofabrik.de/north-america/us/maryland.html (verify: the project documents the file maryland-latest.osm.pbf, replication timestamp 2026-06-11, but not the URL)
- How: Download maryland-latest.osm.pbf. Used only indirectly: it is the source of the MATSim network (4_matsim/network: pt2matsim OSM converter, build_networks.py), on which the transit schedule is mapped and the period car times are measured.
- Put in: 00_shared_data/raw/osm
- Files: `maryland-latest.osm.pbf`

**MATSim 2024 network with transit links and the regional transit schedule** — this project: 4_matsim/network (build_networks.py) and 4_matsim/transit (build_transit.sh: pt2matsim 24.4 Gtfs2TransitSchedule + PublicTransitMapper)
- Where: not downloadable: project product (pt2matsim: https://github.com/matsim-org/pt2matsim, verify)
- How: Copy 4_matsim/transit/network_2024_pt.xml.gz (and, for the door-to-door skims, the mapped schedule 4_matsim/transit/transitSchedule_2024* and vehicles) of the project.
- Put in: 00_shared_data/raw/matsim_network_2024
- Files: `network_2024_pt.xml.gz`, `transitSchedule_2024.xml.gz`, `transitVehicles_2024.xml.gz`

**MATSim resident-only assignment of the chain (events and network), source of the car time profile by period** — this project: Updated MATSim/I695_pricing_runs/FIT/R6 (2023 network, Key Bridge open, 20 iterations)
- Where: not downloadable: project product
- How: Copy output_events.xml.gz and output_network.xml.gz of that run.
- Put in: 00_shared_data/raw/matsim_resident_run_2023
- Files: `output_events.xml.gz`, `output_network.xml.gz`

**MATSim 2024.0 run jar of the project and compiled TransitSkim classes** — this project: Updated MATSim/code/matsim-run (baltimore-matsim-1.0.jar); pipeline/java/TransitSkim.java
- Where: not downloadable: project build (MATSim: https://www.matsim.org, verify)
- How: Copy Updated MATSim/code/matsim-run/target/baltimore-matsim-1.0.jar; compile TransitSkim.java (copy in 00_shared_prep/code) against it with javac 21 into classes/ (transit_skim.py does this) and tsclasses/.
- Put in: 00_shared_data/raw/matsim_tools
- Files: `baltimore-matsim-1.0.jar`, `classes/`, `tsclasses/`

**ACS PUMS 2015-2019 5-year person files of DC, MD, VA, WV** — U.S. Census Bureau
- Where: https://www2.census.gov/programs-surveys/acs/data/pums/2019/5-Year/csv_p<st>.zip
- How: Download csv_pdc.zip, csv_pmd.zip, csv_pva.zip, csv_pwv.zip. wesml_targets.py keeps interview years 2017-2019 (SERIALNO), workers at work (ESR 1, 4, age 16+) not working at home (JWTRNS present, not 11), weights PWGTP; commute mode JWTRNS, car occupancy JWRIP; geography PUMA (2010).
- Put in: 00_shared_data/raw/acs_pums_2015_2019_5yr
- Files: `csv_pdc.zip`, `csv_pmd.zip`, `csv_pva.zip`, `csv_pwv.zip`

**Census 2010 tract-to-PUMA relationship file** — U.S. Census Bureau
- Where: https://www2.census.gov/geo/docs/maps-data/data/rel/2010_Census_Tract_to_2010_PUMA.txt
- How: Download the file as is.
- Put in: 00_shared_data/raw/census_tract_to_puma_2010
- Files: `2010_Census_Tract_to_2010_PUMA.txt`

**Paid-parking share of car-driver tours by destination area** — derived from the RTS (field park_pay of RTS_raw_flat_data.csv): prep_tours.py
- Where: no download (survey-derived)
- How: prep_tours.py: share of estimation-half car-driver tours whose driver paid for parking, weighted, by destination area class (Baltimore City CBD / rest, DC CBD / rest, other CBD, rest).
- Put in: 00_shared_data/mode_choice_inputs/parking_paid_share.csv (provided)
- Files: `parking_paid_share.csv`

**AAA 'Your Driving Costs' 2017: operating cost of the average new vehicle at 15,000 mi/yr** — AAA
- Where: https://newsroom.aaa.com/auto/your-driving-costs/ (verify: the project cites the 2017 edition, 0.179 $/mile = fuel 10.0 + maintenance and repair 7.9 cents, without a URL)
- How: One number, coded in mc_common.py (OPC_KM = 0.179 / 1.60934 $/km).
- Put in: none (constant in the code)

**Transit fares 2017/18: MTA $1.80, Metrobus $2.00, WMATA Metrorail by distance and period, MARC by distance** — MTA Maryland, WMATA
- Where: https://www.mta.maryland.gov/ and https://www.wmata.com/ (verify: fare_rule in mc_common.py has no source URL)
- How: Coded in mc_common.fare_rule.
- Put in: none (rule in the code)

## 5. Model specification

*The utility functions, read from the model script; nothing is re-specified here.*

The folder's script `model/estimate_tour_mode_work.py`, imported as it is. Its utility functions:

In [ ]:
# Model specification
sys.path.insert(0, os.path.join(FOLDER, "model")); sys.modules.pop("estimate_tour_mode_work", None)   # the model folder's script (input/code may hold a namesake)
import estimate_tour_mode_work as M
src = inspect.getsource(M)
print(src[:src.find("def read(")])

In [ ]:
# Model specification
r = M.read("estimation")
df = r[0] if isinstance(r, tuple) else r
print(f"estimation table as read by the script: {df.shape[0]:,} rows, {df.shape[1]} columns")

## 6. Correlation of the explanatory variables

*Correlation of the explanatory variables. Look for: |r| of 0.7 or more and VIF of 10 or more; these make single coefficients less precise.*

|r| ≥ 0.7 and VIF ≥ 10 are listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
# Correlation of the explanatory variables
f = M.formulas() if hasattr(M, "formulas") else M.loglike()
expr = f["loglike"] if isinstance(f, dict) else f
used = [c for c in nt.model_variables(expr) if c in df and not c.startswith("av_") and c not in ("y", "wesml", "w", "weight")]
Xu = nt.explanatory(df[used])
print(f"{len(used)} data columns enter the utilities")
Cu = nt.corr_heatmap(Xu, "Variables of the utilities")
display(nt.high_pairs(Cu)); display(nt.vif(Xu))

## 7. Log-likelihood at the reference coefficients

*Log-likelihood at the article coefficients, without estimation. Look for: the two numbers should be equal (to rounding).*

No estimation: the script's `evaluate("estimation", "full")`, i.e. the log-likelihood of the estimation half at the full-precision coefficients
of `model/reference_estimates.csv`. Target: LL(beta) of that file, -4313.096850944428 (article -4313.1;
N = 9126, K = 51).

In [ ]:
# Log-likelihood at the reference coefficients
LL_BETA = -4313.096850944428
def evaluate(half):
    here = os.getcwd()
    try:
        return M.evaluate(half, 'full') if "at" in inspect.signature(M.evaluate).parameters else M.evaluate(half)
    finally:
        os.chdir(here)

if RUN_CHECK:
    t0 = time.time(); ll_ref = evaluate("estimation")
    print(f"{(time.time() - t0) / 60:.1f} min")
    if isinstance(ll_ref, (int, float)) and LL_BETA is not None:
        print(f"LL at the reference coefficients {ll_ref:,.4f}; LL(beta) of reference_estimates.csv {LL_BETA:,.4f}; difference {ll_ref - LL_BETA:+.4f}")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

*Biogeme estimation, only with RUN_ESTIMATION. Look for: convergence and the run time.*

The script's `estimate()`: Biogeme with its defaults (`output/biogeme.toml`), robust standard errors. Report in
`output/eval.html` and `.yaml`. Not run here (`RUN_ESTIMATION = False`); a result in `output/` is read only if it is
newer than `model/reference_estimates.csv`.

In [ ]:
# Biogeme estimation
if RUN_ESTIMATION:
    here = os.getcwd(); t0 = time.time()
    try:
        M.estimate(OUT)
    finally:
        os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s")
ref_time = os.path.getmtime(os.path.join(FOLDER, "model", "reference_estimates.csv"))
y = [f for f in sorted(glob.glob(os.path.join(OUT, "*.yaml")), key=os.path.getmtime) if "eval" not in os.path.basename(f) and os.path.getmtime(f) > ref_time]
results = EstimationResults.from_yaml_file(y[-1]) if y else None
print(f"results read from {os.path.basename(y[-1])}" if y else "no current result in output/: the estimate tables of section 9 are skipped")

## 9. Results

*The estimates and what they mean. Look for: coefficients equal to the article within rounding, and the readings below.*

### Estimates in the article format
Coefficient (stars: *** p < 0.01, ** p < 0.05, * p < 0.10, as in the article), robust s.e., p.

In [ ]:
# Results
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### What the coefficients say

One plain line per term (article estimates of `model/reference_estimates.csv`) and the coefficient plot.

In [ ]:
# plain readings of the coefficients and the coefficient plot (95 % robust intervals, grouped by block)
rows = af.coef_rows(MODEL_FOLDER)
display(rd.abit_readings(rows).style.hide(axis="index").set_properties(**{"text-align": "left"}))
fg.coef_figure(rows); plt.show()

How to read it: a point right of zero makes the alternative more likely; an interval that crosses zero is not significant; colours are the alternatives or purposes a term belongs to.

### Goodness of fit

In [ ]:
# Goodness of fit
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
# Comparison with the article (model/reference_estimates.csv)
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "K": results.number_of_free_parameters, "N": 9126})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
# Correlation of the estimates
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

*Fit on the hold-out records, which the estimation did not use. Look for: a log-likelihood per record close to that of the estimation sample, and the maps.*

Log-likelihood of the hold-out half at the reference coefficients (`evaluate("holdout", "full")`).

In [ ]:
# Hold-out sample
if RUN_HOLDOUT:
    ll_hold = evaluate("holdout")
else:
    print("skipped (RUN_HOLDOUT = False)")

### Hold-out shares

Observed and model shares of the hold-out half (model: mean of the probabilities at the reference coefficients, simulated with this script's own log-likelihood).

In [ ]:
# observed against model shares on the hold-out half, overall and for one segment
t1, (title, t2, xl), n_hold = af.holdout_tables(MODEL_FOLDER)
t2 = t2[(t2.observed > 0.01) | (t2.model > 0.01)]
A = af.ALTS[MODEL_FOLDER]
fg.obs_model_figure([("All" if len(A) > 2 else t1.index[0], t1, "", "Hold-out records (%)"), (title, t2, xl, f"{t1.index[0] if len(A) == 2 else A[1]} (%)")]); plt.show()
print(f"hold-out records: {n_hold:,}")

How to read it: dark grey observed, orange model; equal bars mean the model reproduces the hold-out shares overall and within the segment. The hold-out half was also used in the specification search.

## Glossary

- **MAST**: Maryland Activity Scheduling and Travel model: the activity-based model of the MILES platform, built on the ABIT code base (Activity-Based Incremental Transport model).
- **reference alternative**: The alternative whose utility is fixed at 0; the coefficients of the others are differences against it.
- **log-likelihood**: Sum over decision makers of ln(probability of the observed choice). Closer to 0 is better.
- **hold-out**: The 20 % (residential, cars) or half (survey models) of the records not used for estimation. It was also used in the specification search, so its fit is not a fully independent test.
- **robust s.e.**: Standard error from the sandwich formula, valid when the model is not exactly right.
- **VIF**: Variance inflation factor: how much the correlation with other variables inflates a coefficient's variance; 10 or more is high.